**Статус: полная валидация остановлена пользователем; результаты 256 точек на 640 эрах не получены.**

# Numerai: 128 и 256 опорных точек RFR LHS

Повторное сравнение на тех же **114 800 обучающих строках**, **42 признаках** и
**640 валидационных эрах**. В RFR меняется только число опорных точек: 128 → 256.
Исходная Kf, C=1, n=t=100000, sig=0, alpha=0.1, seed=42, свободный член и масштаб
признаков сохранены. Новый набор LHS генерируется целиком; он не является
добавлением ещё 128 точек к прежнему набору.

Обучающая выборка взята из сохранённого первого запуска RFR и проверена по хешу
идентификаторов, а также повторным обучением Ridge. Результаты предыдущих моделей
сохранены отдельно. Официальный CORR вычисляется для каждой эры с равными весами.

Валидация уже использовалась в предыдущих экспериментах. Это исследовательское
сравнение числа опорных точек; при выборе параметров понадобится новая
окончательная оценка. Накопленный CORR — не доходность портфеля.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'experiments' / 'numerai_baselines.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'experiments' / 'numerai_baselines.py').exists()
RESULTS = ROOT / 'experiments' / 'results' / 'numerai_rfr_256'
plt.rcParams['figure.figsize'] = (12, 4)


## Повторный запуск

Ноутбук по умолчанию показывает готовые результаты. Для повторения выбери
интерпретатор `.venv/Scripts/python.exe` и установи `RUN_TRAINING=True`.

```powershell
.\.venv\Scripts\python.exe experiments\numerai_rfr.py --n-support 256
```

Файлы запуска на 256 точках сохраняются отдельно от запуска на 128 точках.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    sys.path.insert(0, str(ROOT / 'experiments'))
    from numerai_rfr import run
    run(n_support=256)


In [ ]:
run_info = json.loads((RESULTS / 'run_config.json').read_text(encoding='utf-8'))
config = run_info['baseline_config'].copy()
prior_info = json.loads((ROOT / 'experiments/results/numerai_rfr/run_config.json').read_text())
config['fit_seconds'] = dict(config['fit_seconds'], RFR_LHS_128=prior_info['fit_seconds'], RFR_LHS_256=run_info['fit_seconds'])
summary = pd.read_csv(RESULTS / 'summary.csv', index_col='model')
scores = pd.read_csv(RESULTS / 'per_era_corr.csv', index_col='era')
periods = pd.read_csv(RESULTS / 'period_summary.csv')
display(pd.Series({
    'target': config['target'],
    'features': config['feature_count'],
    'train_rows': config['training']['rows'],
    'train_eras': config['training']['era_count'],
    'train_era_range': (config['training']['first_era'], config['training']['last_era']),
    'validation_rows': config['validation']['rows'],
    'validation_eras': config['validation']['era_count'],
    'validation_era_range': (config['validation']['first_era'], config['validation']['last_era']),
    'skipped_validation_rows': config['validation']['skipped'],
    'fit_seconds': config['fit_seconds'],
}))
display(summary.round(6))

display(pd.Series(run_info['rfr_parameters']))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
scores[['Ridge', 'LightGBM', 'RFR_LHS_128', 'RFR_LHS_256']].cumsum().plot(ax=axes[0])
axes[0].set_ylabel('Cumulative CORR')
axes[0].set_title('Cumulative per-era Numerai CORR (not portfolio returns)')
scores[['Ridge', 'LightGBM', 'RFR_LHS_128', 'RFR_LHS_256']].rolling(52, min_periods=52).mean().plot(ax=axes[1])
axes[1].axhline(0, color='black', linewidth=0.7)
axes[1].set_ylabel('Mean CORR, 52 eras')
axes[1].set_xlabel('Era')
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS / 'corr_over_time.png', dpi=150)
plt.show()


## Стабильность во времени

Две половины валидации показаны отдельно. Это описательная проверка, а не наборы
для настройки параметров. Соседние эры имеют перекрывающиеся целевые горизонты:
их нельзя считать независимыми для простых оценок статистической значимости.

In [ ]:
display(periods.round(6))
fig, ax = plt.subplots(figsize=(10, 4))
scores[['Ridge', 'LightGBM', 'RFR_LHS_128', 'RFR_LHS_256']].plot.hist(bins=40, alpha=0.5, ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Per-era Numerai CORR')
ax.set_title('Distribution of per-era scores')
fig.tight_layout()
plt.show()


## Разница качества и затрат

Сравниваем средний CORR, CORR Sharpe, долю положительных эр и стабильность по
двум половинам периода. Ниже приведены абсолютная разница и относительное
изменение среднего CORR, а также измеренное время этапов нового запуска.

Первый запуск на 128 точках не измерял чистый инференс отдельно. Его полное время
нельзя напрямую сравнивать с чистым временем предсказаний нового запуска.

Результаты: `experiments/results/numerai_rfr_256/`.
Модель и прогнозы: `downloads/numerai/rfr_artifacts_256/`.


In [ ]:
a = summary.loc['RFR_LHS_128', 'mean_corr']
b = summary.loc['RFR_LHS_256', 'mean_corr']
display(pd.Series({
    'mean_corr_difference_256_minus_128': b - a,
    'mean_corr_relative_change_percent': (b / a - 1) * 100,
    'fit_seconds_128': prior_info['fit_seconds'],
    'fit_seconds_256': run_info['fit_seconds'],
    'total_seconds_128': prior_info['total_seconds'],
    'total_seconds_256': run_info['total_seconds'],
}))
display(pd.Series(run_info['stage_timings']))
